In [ ]:
import sympy as sp
from itertools import combinations, permutations

def naive_permanent(M):
    """Calculates the permanent of a matrix."""
    n = len(M)
    if n == 0:
        return sp.S.One

    perm_sum = 0
    for p in permutations(range(n)):
        term = 1
        for i in range(n):
            if M[i][p[i]] == 0:
                term = 0
                break
            term *= M[i][p[i]]
        perm_sum += term
    return perm_sum

def evaluate_residual_submatrices(n_max=8):
    """
    Summarizes the total sum of per(B(alpha|alpha)) for each r to identify patterns.
    """
    c1, c2, c3, c4 = sp.symbols('c1 c2 c3 c4')

    for n in range(4, n_max + 1):
        print(f"\n{'='*60}")
        print(f"Matrix Size n = {n}")
        print(f"{'='*60}")

        B_full = [[0]*n for _ in range(n)]
        for i in range(n):
            B_full[i][(i + 1) % n] = c2
            B_full[i][(i + 2) % n] = c3
            B_full[i][(i + 3) % n] = c4

        for r in range(1, n):
            total_r_sum = 0
            # Generate all subsets alpha of size r
            for alpha in combinations(range(n), r):
                remaining_indices = [i for i in range(n) if i not in alpha]

                B_sub = []
                for i in remaining_indices:
                    row = [B_full[i][j] for j in remaining_indices]
                    B_sub.append(row)

                sub_perm = naive_permanent(B_sub)
                if sub_perm != 0:
                    total_r_sum += sub_perm

            print(f"r = {r} (c1^{r} coeff): {total_r_sum}")

            # Print only the summary for this r
            #simplified_sum = sp.simplify(total_r_sum)
            #rint(f"r = {r} (c1^{r} coeff): {simplified_sum}")

if __name__ == '__main__':
    # Setting n_max=8 to ensure results are returned quickly without interruption
    evaluate_residual_submatrices(n_max=14)

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
from itertools import combinations

def generate_circulant_matrix(n, c1, c2, c3, c4):
    row = [c1, c2, c3, c4] + [0] * (n - 4)
    M = np.zeros((n, n), dtype=object)
    for i in range(n):
        M[i] = np.roll(row, i)
    return M

def ryser_permanent(M):
    n = len(M)
    if n == 0: return 1
    total = 0
    for i in range(1, 1 << n):
        sign = -1 if (n - bin(i).count('1')) % 2 else 1
        prod = 1
        for j in range(n):
            row_sum = 0
            for k in range(n):
                if (i >> k) & 1:
                    row_sum += M[j][k]
            prod *= row_sum
        total += sign * prod
    return total

def explicit_minor_permanent_optimized(n, c1, c2, c3, c4):
    B = generate_circulant_matrix(n, 0, c2, c3, c4)
    total_permanent = 0
    memo = {}

    for r in range(n + 1):
        if r == n:
            total_permanent += c1**n
            continue

        minor_size = n - r
        if minor_size == 0:
            total_permanent += c1**r
            continue

        indices = list(range(n))
        minor_sum = 0

        # We only need to check unique relative patterns of indices
        # because of the circulant symmetry of B.
        for subset in combinations(indices, minor_size):
            # The canonical form for a subset of indices in a circulant graph
            # is often represented by its relative offsets from the first element.
            rel_indices = tuple(sorted([(idx - subset[0]) % n for idx in subset]))

            if rel_indices not in memo:
                sub_B = B[np.ix_(subset, subset)]
                memo[rel_indices] = ryser_permanent(sub_B)

            minor_sum += memo[rel_indices]

        total_permanent += (c1**r) * minor_sum

    return total_permanent

def run_benchmarks_and_plot():
    sizes = list(range(4, 12))
    ryser_times = []
    explicit_times = []
    c1, c2, c3, c4 = 1, 2, 3, 4

    print(f"{'n':<4} | {'Match?':<6} | {'Ryser (s)':<12} | {'Explicit Opt (s)':<12}")
    print('-' * 55)

    for n in sizes:
        M = generate_circulant_matrix(n, c1, c2, c3, c4)

        start_r = time.time()
        res_ryser = ryser_permanent(M)
        t_r = time.time() - start_r
        ryser_times.append(t_r)

        start_e = time.time()
        res_explicit = explicit_minor_permanent_optimized(n, c1, c2, c3, c4)
        t_e = time.time() - start_e
        explicit_times.append(t_e)

        print(f'{n:<4} | {str(res_ryser == res_explicit):<6} | {t_r:<12.5f} | {t_e:<12.5f}')

    plt.figure(figsize=(10, 6))
    plt.plot(sizes, ryser_times, 'ro-', label='Standard Ryser')
    plt.plot(sizes, explicit_times, 'bs-', label='Explicit (Symmetry Optimized)')
    plt.yscale('log')
    plt.xlabel('Matrix Size (n)')
    plt.ylabel('Execution Time (seconds)')
    plt.title('Performance Comparison: Ryser vs Optimized Explicit')
    plt.legend()
    plt.grid(True)
    plt.show()

if __name__ == "__main__":
    run_benchmarks_and_plot()

In [ ]:
import time
import matplotlib.pyplot as plt

def generate_circulant_matrix(n, c1, c2, c3, c4):
    """Generates an n x n circulant matrix safely handling overlap for n < 4."""
    M = [[0] * n for _ in range(n)]
    c = [c1, c2, c3, c4]

    for i in range(n):
        for d in range(4):
            target = (i + d) % n
            M[i][target] += c[d]

    return M

def ryser_permanent(M):
    """Standard Ryser's algorithm for baseline comparison. Time: O(n * 2^n)."""
    n = len(M)
    if n == 1:
        return M[0][0]

    total = 0
    for i in range(1, 1 << n):
        sign = -1 if (n - bin(i).count('1')) % 2 else 1
        prod = 1
        for j in range(n):
            row_sum = sum(M[j][k] for k in range(n) if (i >> k) & 1)
            prod *= row_sum
        total += sign * prod

    return total

def explicit_dp_permanent(n, c1, c2, c3, c4):
    """
    Programmatic evaluator for your explicit Combinatorial Cycle Covers.
    It executes the integer partitions implicitly via state-tracking. Time: O(n).
    """
    c = [c1, c2, c3, c4]
    memo = {}

    def dfs(row, available_mask):
        # Base case: All rows processed, check if all targets are perfectly hit
        if row == n:
            return 1 if available_mask == 0 else 0

        state = (row, available_mask)
        if state in memo:
            return memo[state]

        total = 0
        # Only iterate through the 4 valid explicit steps (0, 1, 2, 3)
        for d in range(4):
            target = (row + d) % n

            # If the target vertex hasn't been hit yet, branch into it
            if available_mask & (1 << target):
                new_mask = available_mask & ~(1 << target)
                total += c[d] * dfs(row + 1, new_mask)

        memo[state] = total
        return total

    # Start at row 0, all n target vertices are available (binary 111...1)
    return dfs(0, (1 << n) - 1)

def run_benchmarks():
    sizes = list(range(1, 22))
    ryser_times = []
    explicit_times = []

    c1, c2, c3, c4 = 1, 2, 3, 4

    print(f"{'n':<4} | {'Match?':<6} | {'Ryser Time (s)':<15} | {'Explicit Time (s)':<15}")
    print("-" * 55)

    for n in sizes:
        M = generate_circulant_matrix(n, c1, c2, c3, c4)

        # 1. Benchmark Ryser
        t0 = time.perf_counter()
        res_ryser = ryser_permanent(M)
        t1 = time.perf_counter()
        ryser_times.append(t1 - t0)

        # 2. Benchmark Explicit Combinatorial Logic
        t0 = time.perf_counter()
        res_explicit = explicit_dp_permanent(n, c1, c2, c3, c4)
        t1 = time.perf_counter()
        explicit_times.append(t1 - t0)

        # 3. Verify Equality
        match = (res_ryser == res_explicit)
        print(f"{n:<4} | {str(match):<6} | {ryser_times[-1]:<15.6f} | {explicit_times[-1]:<15.6f}")

    # ==========================================
    # Generate the Graphical Comparison
    # ==========================================
    plt.figure(figsize=(10, 6))
    plt.plot(sizes, ryser_times, marker='o', color='red', linewidth=2, label="Ryser's Formula $\mathcal{O}(n 2^n)$")
    plt.plot(sizes, explicit_times, marker='s', color='blue', linewidth=2, label="Explicit Combinatorial Logic $\mathcal{O}(n)$")

    plt.yscale('log')
    plt.xlabel('Matrix Size (n)', fontsize=12, fontweight='bold')
    plt.ylabel('Execution Time (seconds) [Log Scale]', fontsize=12, fontweight='bold')
    plt.title('Execution Time: Ryser vs. Explicit Formula (n=1 to 12)', fontsize=14, fontweight='bold')

    plt.xticks(sizes)
    plt.grid(True, which="both", linestyle="--", alpha=0.6)
    plt.legend(fontsize=12, loc='upper left')
    plt.tight_layout()

    plt.savefig('benchmark_1_to_12.png', dpi=300)
    print("\nGraph saved as 'benchmark_1_to_12.png'. Displaying now...")
    plt.show()

if __name__ == "__main__":
    run_benchmarks()